# RQ1 — Content of the files

Reproduces the subsection *Content of the files* of the paper, with the labels of classifier run 1:

| Result in the paper | Where |
|---|---|
| Table of prevalence of the 13 categories (1,828 files) | section 1 |
| Co-occurrence figure (Jaccard) and the `run_and_build` × `test_instructions` pair | section 2 |
| Categories per file, files with a single category (pointers) | section 3 |
| Figure of file size by number of categories | section 4 |

Inputs: `data/rq01/classifier/rq01_classification_run_1.csv` and the files in
`data/collection/context_files/` (for the size in characters).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "collection").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
from figures import save, svg_categories_table, svg_jaccard_column, svg_size

RUN_1 = ROOT / "data/rq01/classifier/rq01_classification_run_1.csv"
OUT = ROOT / "results/rq01"
OUT.mkdir(parents=True, exist_ok=True)

CATEGORIES = [
    "run_and_build", "project_overview", "doc_metadata", "test_instructions",
    "project_structure", "system_design", "contribution_guidelines", "code_conventions",
    "ai_rules", "performance_&_resource_management", "troubleshooting", "security",
    "deployment_&_infrastructure",
]

df = pd.read_csv(RUN_1)
for c in CATEGORIES:
    df[c] = df[c].fillna(0).astype(int)
N = len(df)
print(f"{N} files in {df['repo'].nunique()} repositories")

1828 files in 1238 repositories


## 1. Prevalence of the categories

In [2]:
prev = (pd.DataFrame({"category": CATEGORIES, "files": df[CATEGORIES].sum().values})
        .sort_values("files", ascending=False, kind="stable").reset_index(drop=True))
prev["pct"] = 100 * prev["files"] / N
ORDER = prev["category"].tolist()
prev.to_csv(OUT / "category_prevalence.csv", index=False)
save(svg_categories_table(prev, N), OUT / "categories-table.svg")
prev.round(1)

saved /home/keven/Documents/tcc/repo-pesquisa-repro/results/rq01/categories-table.svg


,category,files,pct
0,test_instructions,1257,68.8
1,code_conventions,1245,68.1
2,run_and_build,1182,64.7
3,ai_rules,1026,56.1
4,project_overview,974,53.3
5,doc_metadata,937,51.3
6,project_structure,912,49.9
7,contribution_guidelines,895,49.0
8,system_design,696,38.1
9,troubleshooting,323,17.7


## 2. Co-occurrence (Jaccard index)

In [3]:
SHORT = {"test_instructions": "test", "code_conventions": "conventions", "run_and_build": "run & build",
         "ai_rules": "ai rules", "project_overview": "overview", "doc_metadata": "doc metadata",
         "project_structure": "structure", "contribution_guidelines": "contribution",
         "system_design": "system design", "troubleshooting": "troubleshooting", "security": "security",
         "deployment_&_infrastructure": "deployment", "performance_&_resource_management": "performance"}
M = df[ORDER].values.astype(bool)
k = len(ORDER)
jac = np.array([[(M[:, a] & M[:, b]).sum() / (M[:, a] | M[:, b]).sum() for b in range(k)] for a in range(k)])
pd.DataFrame(jac, index=ORDER, columns=ORDER).round(3).to_csv(OUT / "category_jaccard.csv")
save(svg_jaccard_column(jac, [SHORT[c] for c in ORDER]), OUT / "jaccard-categories-column.svg")

pairs = pd.DataFrame([(ORDER[i], ORDER[j], jac[i, j]) for i in range(k) for j in range(i)],
                     columns=["cat_a", "cat_b", "jaccard"]).sort_values("jaccard", ascending=False)
build, test = df["run_and_build"] == 1, df["test_instructions"] == 1
print(f"run_and_build × test_instructions: Jaccard = {(build & test).sum() / (build | test).sum():.2f}")
print(f"  both: {(build & test).sum()}; build only: {(build & ~test).sum()}; tests only: {(test & ~build).sum()}")
print(f"  files with build that also have tests: {100 * (build & test).sum() / build.sum():.1f}%")
print(f"  files with tests that also have build: {100 * (build & test).sum() / test.sum():.1f}%")
pairs.head(5).round(2)

saved /home/keven/Documents/tcc/repo-pesquisa-repro/results/rq01/jaccard-categories-column.svg
run_and_build × test_instructions: Jaccard = 0.77
  both: 1059; build only: 123; tests only: 198
  files with build that also have tests: 89.6%
  files with tests that also have build: 84.2%


,cat_a,cat_b,jaccard
1,run_and_build,test_instructions,0.77
0,code_conventions,test_instructions,0.74
2,run_and_build,code_conventions,0.68
8,project_overview,run_and_build,0.63
6,project_overview,test_instructions,0.61


## 3. Categories per file and pointers

In [4]:
df["n_categories"] = df[CATEGORIES].sum(axis=1)


def characters(link):
    path = ROOT / link.replace("local:", "")
    return len(path.read_text(encoding="utf-8", errors="ignore"))


df["characters"] = df["link_arquivo"].map(characters)

print(f"Categories per file: median {df['n_categories'].median():.0f}, mean {df['n_categories'].mean():.1f}")
print(f"Files with 5 to 8 categories: {100 * df['n_categories'].between(5, 8).mean():.1f}%")
print(f"Files with all 13 categories: {(df['n_categories'] == 13).sum()}")

single = df[df["n_categories"] == 1]
doc_only = single[single["doc_metadata"] == 1]
print(f"\nFiles with a single category: {len(single)} ({100 * len(single) / N:.1f}%), "
      f"median of {single['characters'].median():.0f} characters, {100 * (single['characters'] < 100).mean():.0f}% under 100, "
      f"{(single['arquivo_analisado'] == 'CLAUDE.md').sum()} are CLAUDE.md")
print(f"  doc_metadata only: {len(doc_only)} ({100 * len(doc_only) / len(single):.0f}%), "
      f"median of {doc_only['characters'].median():.0f} characters, "
      f"{100 * (doc_only['characters'] < 100).mean():.0f}% under 100, "
      f"{(doc_only['arquivo_analisado'] == 'CLAUDE.md').sum()} are CLAUDE.md")
print(f"  with a single category and 500 characters or more: {(single['characters'] >= 500).sum()}")
largest = single.sort_values("characters", ascending=False).iloc[0]
print(f"  largest: {largest['repo']} / {largest['arquivo_analisado']}, {largest['characters']:,} characters, "
      f"category {[c for c in CATEGORIES if largest[c]]}")

Categories per file: median 6, mean 5.6
Files with 5 to 8 categories: 58.6%
Files with all 13 categories: 0

Files with a single category: 265 (14.5%), median of 46 characters, 59% under 100, 169 are CLAUDE.md
  doc_metadata only: 223 (84%), median of 24 characters, 70% under 100, 165 are CLAUDE.md
  with a single category and 500 characters or more: 30
  largest: celery/celery / COPILOT, 25,325 characters, category ['ai_rules']


## 4. File size by number of categories

In [5]:
svg, medians = svg_size(df["n_categories"].values, df["characters"].values)
save(svg, OUT / "size-by-categories.svg")
medians.to_csv(OUT / "size_by_categories.csv", index=False)
m = medians.set_index("n_categories")["median_characters"]
print(f"Mean increase per category, from 2 to 10 categories: {(m[10] - m[2]) / 8:,.0f} characters")
print(f"Overall median: {df['characters'].median():,.0f} characters")
medians

saved /home/keven/Documents/tcc/repo-pesquisa-repro/results/rq01/size-by-categories.svg
Mean increase per category, from 2 to 10 categories: 1,850 characters
Overall median: 5,047 characters


,n_categories,files,median_characters
0,1,265,46
1,2,72,1007
2,3,88,1508
3,4,125,2867
4,5,208,3774
5,6,317,5687
6,7,305,7174
7,8,242,9228
8,9,128,11276
9,10,47,15807
